
## READ THE BRONZE DATA AND CREATE THE DATAFRAME

In [1]:
df_customers = spark.read.parquet("Files/Bronze/customers.parquet")
# df _customers now is a Spark DataFrame containing parquet data from "Files/Bronze/customers.parquet".
display(df_customers)

StatementMeta(, 7da29b74-5b96-4749-a273-a2f3e205f8a7, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2e3b5a81-f8e3-4ac5-92ca-d3df92d5e2b6)

In [2]:
df_orders = spark.read.parquet("Files/Bronze/orders.parquet")
# df_orders now is a Spark DataFrame containing parquet data from "Files/Bronze/orders.parquet".
display(df_orders)

StatementMeta(, 7da29b74-5b96-4749-a273-a2f3e205f8a7, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f3161491-a79b-429a-b200-3cc6678b2c37)

In [3]:
df_payments = spark.read.parquet("Files/Bronze/payments.parquet")
# df_payments now is a Spark DataFrame containing parquet data from "Files/Bronze/payments.parquet".
display(df_payments) 

StatementMeta(, 7da29b74-5b96-4749-a273-a2f3e205f8a7, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8dc0a996-6fc6-4729-b844-d0b6f5650483)

In [4]:
df_support_tickets = spark.read.parquet("Files/Bronze/support_tickets.parquet")
# df_support_tickets  now is a Spark DataFrame containing parquet data from "Files/Bronze/support_tickets.parquet".
display(df_support_tickets )

StatementMeta(, 7da29b74-5b96-4749-a273-a2f3e205f8a7, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 321e89fe-316b-4648-b1b1-6b343dd45d90)

In [5]:
df_web_activities = spark.read.parquet("Files/Bronze/web_activities.parquet")
# df_web_activities now is a Spark DataFrame containing parquet data from "Files/Bronze/web_activities.parquet".
display(df_web_activities)

StatementMeta(, 7da29b74-5b96-4749-a273-a2f3e205f8a7, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 289c23a2-cdd4-44dd-9265-7ae1f6143252)

## Clean the Customers file

In [6]:
from pyspark.sql.functions import col, initcap, lower, trim, when, regexp_replace, to_date, coalesce, lit, upper

# 1. CLEAN AND STANDARDIZE THE COLUMNS BASED ON THE NEW IMAGE SCHEMA
df_silver_customers = df_customers \
    .withColumn("CustomerID", col("customer_id").cast("integer")) \
    .withColumn("CustomerName", initcap(trim(col("name")))) \
    .withColumn("Email", lower(trim(col("EMAIL")))) \
    .withColumn("Gender", upper(trim(col("gender")))) \
    .withColumn("Location", initcap(trim(col("location"))))

# 2. HANDLE BLANKS / NULLS
df_silver_customers = df_silver_customers \
    .withColumn("CustomerName", when((col("CustomerName") == "") | col("CustomerName").isNull(), "Unknown").otherwise(col("CustomerName")))

# 3. FIX EMAIL FORMATS (Fix @@ and validate domains)
df_silver_customers = df_silver_customers \
    .withColumn("Email", regexp_replace(col("Email"), "@@", "@")) \
    .withColumn("Email", when(col("Email").rlike(r"^[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}$"), col("Email")).otherwise("Invalid Email"))

# 4. STANDARDIZE GENDER STRINGS
df_silver_customers = df_silver_customers \
    .withColumn("Gender", when(col("Gender").isin("M", "MALE"), "Male") \
                          .when(col("Gender").isin("F", "FEMALE"), "Female") \
                          .otherwise("Unknown"))

# 5. PARSE MIXED DATE FORMATS (Cleans '1/1/1990' or formats hidden behind '#######' or 'not availal')
df_silver_customers = df_silver_customers.withColumn("dob_clean", regexp_replace(col("dob"), "/", "-"))
df_silver_customers = df_silver_customers.withColumn("DateOfBirth", 
    coalesce(
        to_date(col("dob_clean"), "yyyy-MM-dd"),
        to_date(col("dob_clean"), "d-M-yyyy"),
        to_date(col("dob_clean"), "dd-MM-yyyy")
    )
)

# 6. DROP THE OLD RAW WORK COLUMNS AND KEEP THE CLEAN PROPER CASE VERSION
df_silver_customers_final = df_silver_customers.select(
    "CustomerID", 
    "CustomerName", 
    "Email", 
    "Gender", 
    "DateOfBirth", 
    "Location"
)

# 7. SAVE CLEANED DATA AS DELTA TABLE
df_silver_customers_final.write.format("delta").mode("overwrite").saveAsTable("silver_cleaned_customers")

print("Silver Layer table successfully generated with normalized business columns!")


StatementMeta(, 7da29b74-5b96-4749-a273-a2f3e205f8a7, 8, Finished, Available, Finished, False)

Silver Layer table successfully generated with normalized business columns!


In [7]:
display(spark.read.table("silver_cleaned_customers"))


StatementMeta(, 7da29b74-5b96-4749-a273-a2f3e205f8a7, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4c54f11e-023e-47ff-ae87-4a0763d73a55)

In [8]:
display(df_silver_customers_final)

StatementMeta(, 7da29b74-5b96-4749-a273-a2f3e205f8a7, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 177b8fe1-0eae-4ed4-a4de-2fa99152d541)

## Clean the Orders File

In [9]:
from pyspark.sql.functions import col, initcap, trim, when, regexp_replace, to_date, coalesce, abs, lit

# 1. CLEAN IDs AND BASIC STRINGS
df_silver_orders = df_orders \
    .withColumn("OrderID", col("order_id").cast("integer")) \
    .withColumn("CustomerID", col("customer_id").cast("integer")) \
    .withColumn("Status", initcap(trim(col("status"))))

# 2. CLEAN TRANS_AMOUNT (Fix missing, nulls, or accidental negative signs)
df_silver_orders = df_silver_orders \
    .withColumn("Amount", col("amount").cast("double")) \
    .withColumn("Amount", coalesce(col("Amount"), lit(0.0))) \
    .withColumn("Amount", abs(col("Amount"))) # Converts -45 to 45

# 3. UNIFY MIXED DATE FORMATS
# Instead of replacing strings and breaking yyyyMMdd, we test the raw patterns directly.
# Spark's coalesce will stop at the first format that successfully matches.
df_silver_orders = df_silver_orders.withColumn("OrderDate", 
    coalesce(
        to_date(col("order_date"), "yyyy-MM-dd"),
        to_date(col("order_date"), "yyyy/MM/dd"),
        to_date(col("order_date"), "dd-MM-yyyy"),
        to_date(col("order_date"), "dd/MM/yyyy"),
        to_date(col("order_date"), "yyyyMMdd")
    )
)

# 4. SELECT FINAL BUSINESS-READY COLUMNS (Uncommented for a production-ready table)
df_silver_orders_final = df_silver_orders.select(
    "OrderID",
    "CustomerID",
    "OrderDate",
    "Amount",
    "Status"
)

# 5. SAVE TO SILVER LAYER AS A DELTA TABLE
# Note: Saving df_silver_orders_final drops temporary intermediate columns like the old raw columns
df_silver_orders_final.write.format("delta").mode("overwrite").saveAsTable("silver_cleaned_orders")

print("Orders data successfully cleaned and saved to the Silver Layer!")

StatementMeta(, 7da29b74-5b96-4749-a273-a2f3e205f8a7, 11, Finished, Available, Finished, False)

Orders data successfully cleaned and saved to the Silver Layer!


In [10]:
display(spark.read.table("silver_cleaned_orders"))

StatementMeta(, 7da29b74-5b96-4749-a273-a2f3e205f8a7, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b045f575-35df-4e2c-af2f-226511e91428)

## Clean the Payments File

In [11]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

# 1. LOAD THE BRONZE PAYMENTS DATA
# Update path to match your Lakehouse structure (e.g., "Files/Bronze/payments.parquet")
df_payments = spark.read.parquet("Files/Bronze/payments.parquet")

# 2. STANDARDIZE TEXT STRINGS AND STATUS
# - Clean up status fields to formal proper case (e.g., "Success", "Failed")
# - Fill null payment statuses with "Unknown"
df_silver_payments = df_payments \
    .withColumn("PaymentID", F.trim(F.col("payment_id"))) \
    .withColumn("CustomerID", F.col("customer_id").cast("integer")) \
    .withColumn("PaymentStatus", F.initcap(F.trim(F.col("payment_status")))) \
    .fillna({"PaymentStatus": "Unknown"})

# 3. UNIFY INCONSISTENT PAYMENT METHODS
# Maps "creditcard" and "credit card" to a single unified "Credit Card" format, and capitalizes others
df_silver_payments = df_silver_payments.withColumn(
    "PaymentMethod",
    F.when(F.lower(F.trim(F.col("payment_method"))).isin("creditcard", "credit card"), F.lit("Credit Card"))
     .when(F.lower(F.trim(F.col("payment_method"))) == "upi", F.lit("UPI"))
     .otherwise(F.initcap(F.trim(F.col("payment_method"))))
)

# 4. FIX TRANSACTION AMOUNTS (Handle Nulls & Absolute Values for Negatives)
df_silver_payments = df_silver_payments \
    .withColumn("Amount", F.col("amount").cast("double")) \
    .withColumn("Amount", F.coalesce(F.col("Amount"), F.lit(0.0))) \
    .withColumn("Amount", F.abs(F.col("Amount"))) \
    .withColumn("Amount", F.col("Amount").cast(DecimalType(10, 2)))

# 5. UNIFY MIXED DATE FORMATS
# Uses coalesce to parse raw patterns sequentially without needing string replacements
df_silver_payments = df_silver_payments.withColumn(
    "PaymentDate", 
    F.coalesce(
        F.to_date(F.col("payment_date"), "yyyy-MM-dd"),
        F.to_date(F.col("payment_date"), "yyyy/MM/dd"),
        F.to_date(F.col("payment_date"), "dd-MM-yyyy"),
        F.to_date(F.col("payment_date"), "dd/MM/yyyy"),
        F.to_date(F.col("payment_date"), "yyyyMMdd")
    )
)

# 6. SELECT FINAL CLEANED COLUMNS
df_silver_payments_final = df_silver_payments.select(
    "PaymentID",
    "CustomerID",
    "PaymentDate",
    "PaymentMethod",
    "PaymentStatus",
    "Amount"
)

# 7. SAVE AS A MANAGED DELTA TABLE IN THE SILVER LAYER
df_silver_payments_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_cleaned_payments")

print("Payments data successfully cleaned and saved to the Silver Layer table: 'silver_cleaned_payments'")

StatementMeta(, 7da29b74-5b96-4749-a273-a2f3e205f8a7, 13, Finished, Available, Finished, False)

Payments data successfully cleaned and saved to the Silver Layer table: 'silver_cleaned_payments'


In [12]:
display(spark.read.table("silver_cleaned_payments"))

StatementMeta(, 7da29b74-5b96-4749-a273-a2f3e205f8a7, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f60ae6f8-c529-478e-80ca-3484ea529f07)

## Clean the Support Tickets file

In [13]:
from pyspark.sql import functions as F

# 1. LOAD THE BRONZE SUPPORT TICKETS DATA
# Adjust the file path to match your Lakehouse directory structure
df_tickets = spark.read.parquet("Files/Bronze/support_tickets.parquet")

# 2. BASIC STRING & ID STANDARDIZATION
df_silver_tickets = df_tickets \
    .withColumn("TicketID", F.trim(F.col("ticket_id"))) \
    .withColumn("CustomerID", F.col("customer_id").cast("integer"))

# 3. CLEAN & UNIFY ISSUE TYPES
# - Trim whitespace and make it proper title case
# - Replace nulls or placeholder text like "Na" with "Unknown"
# - Unify highly similar overlapping categories for clean downstream grouping
df_silver_tickets = df_silver_tickets \
    .withColumn("IssueType", F.initcap(F.trim(F.col("issue_type")))) \
    .fillna({"IssueType": "Unknown"}) \
    .withColumn("IssueType", 
        F.when(F.col("IssueType").isin("Na", "Nan", ""), F.lit("Unknown"))
         .when(F.col("IssueType") == "Payment Error", F.lit("Payment Issue"))
         .when(F.col("IssueType") == "Login Problem", F.lit("Login Issue"))
         .when(F.col("IssueType") == "Refund Request", F.lit("Refund"))
         .otherwise(F.col("IssueType"))
    )

# 4. UNIFY MIXED TICKET DATE FORMATS
# Coalesce tests each format string configuration dynamically
df_silver_tickets = df_silver_tickets.withColumn(
    "TicketDate", 
    F.coalesce(
        F.to_date(F.col("ticket_date"), "yyyy-MM-dd"),
        F.to_date(F.col("ticket_date"), "yyyy/MM/dd"),
        F.to_date(F.col("ticket_date"), "dd-MM-yyyy"),
        F.to_date(F.col("ticket_date"), "dd/MM/yyyy"),
        F.to_date(F.col("ticket_date"), "yyyyMMdd")
    )
)

# 5. STANDARDIZE RESOLUTION STATUS
# - Fixes mixed casing (e.g., "closed" vs "Closed" -> "Closed")
# - Flags missing/null statuses as "Unknown" (or "Open")
df_silver_tickets = df_silver_tickets \
    .withColumn("ResolutionStatus", F.initcap(F.trim(F.col("resolution_status")))) \
    .fillna({"ResolutionStatus": "Unknown"}) \
    .withColumn("ResolutionStatus", 
        F.when(F.col("ResolutionStatus").isin("", "Na", "Nan"), F.lit("Unknown"))
         .otherwise(F.col("ResolutionStatus"))
    )

# 6. SELECT FINAL CLEANED COLUMNS
df_silver_tickets_final = df_silver_tickets.select(
    "TicketID",
    "CustomerID",
    "TicketDate",
    "IssueType",
    "ResolutionStatus"
)

# 7. SAVE TO SILVER LAYER AS A DELTA TABLE
df_silver_tickets_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_cleaned_tickets")

print("Support tickets successfully cleaned and saved to the Silver Layer table: 'silver_cleaned_tickets'")

StatementMeta(, 7da29b74-5b96-4749-a273-a2f3e205f8a7, 15, Finished, Available, Finished, False)

Support tickets successfully cleaned and saved to the Silver Layer table: 'silver_cleaned_tickets'


In [14]:
display(spark.read.table("silver_cleaned_tickets"))

StatementMeta(, 7da29b74-5b96-4749-a273-a2f3e205f8a7, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 351db665-0660-41f9-8854-c963fa7c10a6)

## Clean Web Activities File

In [15]:
from pyspark.sql import functions as F

# 1. LOAD THE BRONZE WEB ACTIVITIES DATA
# Adjust the file path to match your Lakehouse directory structure
df_web_activities = spark.read.parquet("Files/Bronze/web_activities.parquet")

# 2. BASIC ID STRIP AND TYPING
df_silver_web = df_web_activities \
    .withColumn("SessionID", F.trim(F.col("session_id"))) \
    .withColumn("CustomerID", F.col("customer_id").cast("integer"))

# 3. STANDARDIZE PAGE VIEW PATHS
# Forces all web paths to lowercase so '/Home' and '/home' aggregate together properly
df_silver_web = df_silver_web.withColumn("PageViewed", F.lower(F.trim(F.col("page_viewed"))))

# 4. UNIFY MIXED DATE FORMATS FOR SESSION TIME
# Safely parses various standard representations into a clean Spark DateType
df_silver_web = df_silver_web.withColumn(
    "SessionDate", 
    F.coalesce(
        F.to_date(F.col("session_time"), "yyyy-MM-dd"),
        F.to_date(F.col("session_time"), "yyyy/MM/dd"),
        F.to_date(F.col("session_time"), "dd-MM-yyyy"),
        F.to_date(F.col("session_time"), "dd/MM/yyyy"),
        F.to_date(F.col("session_time"), "yyyyMMdd")
    )
)

# 5. STANDARDIZE DEVICE TYPES WITH PREMIUM CASING MATCH
# Explicitly handles specific industry mobile/desktop naming structures like 'iOS' and 'macOS'
df_silver_web = df_silver_web.withColumn(
    "DeviceType",
    F.when(F.lower(F.trim(F.col("device_type"))) == "ios", F.lit("iOS"))
     .when(F.lower(F.trim(F.col("device_type"))) == "macos", F.lit("macOS"))
     .otherwise(F.initcap(F.trim(F.col("device_type"))))
)

# 6. SELECT FINAL BUSINESS-READY COLUMNS
df_silver_web_final = df_silver_web.select(
    "SessionID",
    "CustomerID",
    "PageViewed",
    "SessionDate",
    "DeviceType"
)

# 7. SAVE TO SILVER LAYER AS A DELTA TABLE
df_silver_web_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_cleaned_web_activities")

print("Web activities data successfully cleaned and saved to the Silver Layer table: 'silver_cleaned_web_activities'")

StatementMeta(, 7da29b74-5b96-4749-a273-a2f3e205f8a7, 17, Finished, Available, Finished, False)

Web activities data successfully cleaned and saved to the Silver Layer table: 'silver_cleaned_web_activities'


In [16]:
display(spark.read.table("silver_cleaned_web_activities"))

StatementMeta(, 7da29b74-5b96-4749-a273-a2f3e205f8a7, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4600f211-ea58-4698-af4d-94f762ad6880)

## GOLD TABLES - AGGREGATE TABLES

In [19]:
from pyspark.sql import functions as F

# 1. READ ILVER TABLES
df_customers = spark.read.table("silver_cleaned_customers")
df_orders = spark.read.table("silver_cleaned_orders")
df_payments = spark.read.table("silver_cleaned_payments")
df_tickets = spark.read.table("silver_cleaned_tickets")
df_web = spark.read.table("silver_cleaned_web_activities")

# 2. PREPARE SILVER COLUMNS TO AVOID CONFLICTS
df_orders_prep = df_orders.withColumnRenamed("Amount", "OrderAmount").withColumnRenamed("Status", "OrderStatus")
df_payments_prep = df_payments.withColumnRenamed("Amount", "PaymentAmount")

# 3. PERFORM SEQUENTIAL LEFT JOINS
# We start with the master customer list and bring in other tables side-by-side.
# If a customer doesn't have a record in a table, Spark automatically fills it with null.
gold_customer_360 = df_customers \
    .join(df_orders_prep, on="CustomerID", how="left") \
    .join(df_payments_prep, on="CustomerID", how="left") \
    .join(df_tickets, on="CustomerID", how="left") \
    .join(df_web, on="CustomerID", how="left")

# 4. REORDER COLUMNS FOR CLEAN BUSINESS VIEW
df_gold_customer360 = gold_customer_360.select(
    "CustomerID", "CustomerName","Email" ,"Gender" ,"DateofBirth","Location",
    # Order Details
    "OrderID", "OrderDate", "OrderAmount", "OrderStatus",
    # Payment Details
    "PaymentID", "PaymentDate", "PaymentAmount", "PaymentMethod", "PaymentStatus",
    # Support Tickets
    "TicketID", "TicketDate", "IssueType", "ResolutionStatus",
    # Web Traffic
    "SessionID", "SessionDate", "PageViewed", "DeviceType"
)

# Display the merged result inside your notebook
display(df_gold_customer360)

# 5. SAVE TO GOLD LAYER AS A DELTA TABLE
df_gold_customer360.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_customer360")

print("Successfully left-joined all data horizontally into 'gold_customer360'!")

StatementMeta(, 7da29b74-5b96-4749-a273-a2f3e205f8a7, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f45c9887-634f-4eed-ba4f-c34e75d7861e)

Successfully left-joined all data horizontally into 'gold_customer360'!
